# PCA con Prince: Analisis completo

Este notebook implementa un flujo estructurado de PCA con `prince` e incluye:

1. Plano principal (individuos)
2. Circulo de correlaciones
3. Varianza explicada (barras + acumulada)
4. Grafico de sobreposicion (biplot)
5. Contribucion de variables
6. Cosenos al cuadrado (`cos2`)

> Puedes reemplazar el dataset por uno propio manteniendo la misma estructura.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import make_blobs

from scripts.data_prep import standardize_features
from scripts.pca_model import fit_prince_pca, extract_pca_results
from scripts.plot_plano_principal import plot_plano_principal
from scripts.plot_correlacion import plot_circulo_correlaciones
from scripts.plot_varianza import plot_varianza_explicada
from scripts.plot_sobreposicion import plot_biplot_sobreposicion
from scripts.metrics_contrib_cos2 import (
    compute_contribuciones,
    plot_contribuciones_pc1_pc2,
    compute_cos2_variables,
    plot_heatmap_cos2,
)

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

## 1) Carga y preparacion de datos

Usamos un dataset de **clientes** (segmentacion comercial) con **8 variables** de comportamiento.

In [ ]:
# Dataset de clientes (simulado, estilo negocio)
# 8 variables numericas para analisis de segmentacion
X_base, clusters = make_blobs(
    n_samples=600,
    n_features=8,
    centers=3,
    cluster_std=1.8,
    random_state=42,
)

feature_cols = [
    "edad",
    "ingreso_mensual",
    "antiguedad_meses",
    "recencia_dias",
    "frecuencia_compra",
    "ticket_promedio",
    "uso_digital",
    "satisfaccion",
]

df = pd.DataFrame(X_base, columns=feature_cols)

# Escalamos cada variable para que parezcan metricas de negocio realistas
scales = np.array([12, 2200, 18, 25, 7, 120, 15, 10])
shifts = np.array([40, 4500, 36, 30, 8, 180, 55, 70])
df[feature_cols] = df[feature_cols] * scales + shifts

# Etiqueta de segmento de cliente (solo para colorear graficos)
segment_names = {0: "Segmento A", 1: "Segmento B", 2: "Segmento C"}
df["segmento_cliente"] = pd.Series(clusters).map(segment_names)
y = df["segmento_cliente"].copy()

# Estandarizacion (modulo reutilizable)
X_scaled_df, scaler = standardize_features(df, feature_cols)

print("Dimensiones de X:", X_scaled_df.shape)
print("Variables usadas:", list(feature_cols))
X_scaled_df.head()

## 2) Ajuste del modelo PCA con Prince

In [ ]:
pca = fit_prince_pca(X_scaled_df, n_components=4, random_state=42)
results = extract_pca_results(pca, X_scaled_df)

row_coords = results["row_coords"]
col_corr = results["col_corr"]
eigenvalues = results["eigenvalues"]
explained = results["explained"]
cumulative = results["cumulative"]
variance_df = results["variance_df"]

variance_df

## 3) Plano principal (PC1 vs PC2)

In [ ]:
plot_plano_principal(row_coords, y, explained)

## 4) Grafico de correlacion (circulo de correlaciones)

In [ ]:
plot_circulo_correlaciones(col_corr, explained)

## 5) Varianza explicada (barras + acumulada)

In [ ]:
plot_varianza_explicada(explained, cumulative)
variance_df

## 6) Grafico de sobreposicion (biplot)

Se combinan los individuos (scores) y las variables (loadings/correlaciones).

In [ ]:
plot_biplot_sobreposicion(row_coords, col_corr, y, explained)

## 7) Contribucion de las variables

Aproximacion clasica: `contribucion_{j,k} = (corr_{j,k}^2 / eigenvalor_k) * 100`.

In [ ]:
contrib = compute_contribuciones(col_corr, eigenvalues)
contrib

In [ ]:
plot_contribuciones_pc1_pc2(contrib)

## 8) Cosenos al cuadrado (cos2) de variables

Para variables, `cos2` se obtiene como la correlacion al cuadrado con cada componente.

In [ ]:
cos2_vars = compute_cos2_variables(col_corr)
cos2_vars

In [ ]:
plot_heatmap_cos2(cos2_vars)

## 9) Interpretacion rapida

- Componentes con mayor `% Varianza` resumen mejor la informacion.
- En el circulo de correlaciones, vectores largos indican mejor representacion por PC1-PC2.
- En contribuciones, variables con mayor `%` son mas influyentes en cada componente.
- En `cos2`, valores altos indican buena calidad de representacion en ese eje.

---

Si quieres usar tu propio dataset, cambia la seccion de carga y deja el resto igual (siempre que las columnas de entrada sean numericas).